In [1]:
# Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

# Sklearn
import sklearn
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.linear_model import LinearRegression

# Skforecast
import skforecast
from skforecast.recursive import ForecasterRecursive
from skforecast.direct import ForecasterDirect
from skforecast.model_selection import TimeSeriesFold, grid_search_forecaster, backtesting_forecaster
from skforecast.preprocessing import RollingFeatures
from skforecast.utils import save_forecaster, load_forecaster
from skforecast.metrics import calculate_coverage
from skforecast.plot import plot_prediction_intervals

C:\Users\savil\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Possible actions (reward_prob, punishment_prob)
ACTIONS = [
    (0.1, 0.9),
    (0.2, 0.8),
    (0.3, 0.7),
    (0.4, 0.6),
    (0.5, 0.5),
    (0.6, 0.4),
    (0.7, 0.3),
    (0.8, 0.2),
    (0.9, 0.1)
]

print("Possible actions (reward_prob, punishment_prob):", ACTIONS)

Possible actions (reward_prob, punishment_prob): [(0.1, 0.9), (0.2, 0.8), (0.3, 0.7), (0.4, 0.6), (0.5, 0.5), (0.6, 0.4), (0.7, 0.3), (0.8, 0.2), (0.9, 0.1)]


In [3]:
# datafframe_creation(n_participants, n_groups, days) 

# This function creates a dataframe simulating the daily steps of participants, where each participant is assigned to a group. 
# The dataframe includes the date, participant ID, group ID (g01 or g02), baseline steps, and actual steps taken each day for a specified number of days.


def dataframe_creation(n_participants, n_groups, days):

    # Generate IDs for participants and groups
    participants = [f"p{i:02d}" for i in range(1, n_participants + 1)]
    groups = [f"g{i:02d}" for i in range(1, n_groups + 1)]

    # Assign 15 participants to each group
    participants_per_group = 15
    group_assignments = {groups[i]: participants[i*participants_per_group:(i+1)*participants_per_group] for i in range(n_groups)}

    data = []

    for gid, plist in group_assignments.items():
        for pid in plist:
            # Baseline daily behaviour
            baseline = int(np.clip(np.random.normal(6000, 3500), 1000, 15000))

            date_range = pd.date_range("2025-01-01", periods=days, freq="D")

            for day, date in enumerate(date_range):
                baseline_today = int(baseline + np.random.normal(0, 300))
                baseline_today = int(np.clip(baseline_today, 1000, 15000))

                steps = baseline_today + np.random.normal(0, baseline_today * 0.20)
                steps = int(np.clip(steps, 500, 15000))

                data.append({
                    "date": date,
                    "participant_id": pid,
                    "group_id": gid,
                    "baseline": baseline_today,
                    "steps": steps
                })

    # Final dataset
    df = pd.DataFrame(data)
    return df

In [ ]:
# Function: train a time series forecasting model for a specific participant using the provided dataframe.of steps. 
# It uses the skforecast library to perform backtesting and returns the mean squared error and predictions for the specified number of steps ahead.

def model_training_prediction(df, p, regressor, lags, steps, n_days):

    #disabilita warning SettingWithCopyWarning
    pd.options.mode.chained_assignment = None  # default='warn'
    df = df[df['participant_id'] == p].copy()
    df = df.copy()
    df['date'] = pd.to_datetime(df['date'])
    df = df.set_index('date')  
    df = df.asfreq('D')  
    initial_train_size = n_days

    # print(df.tail())

    cv = TimeSeriesFold(
        steps              = steps,
        initial_train_size = initial_train_size,
        window_size        = 7,        
        fixed_train_size   = False,   # keep training size constant
        refit              = True,   # refit the model at each split
    )

    forecaster = ForecasterRecursive(
                regressor = regressor,
                lags      = lags
             )


    metric, predictions = backtesting_forecaster(
                                    forecaster = forecaster,
                                    y          = df['steps'],
                                    cv         = cv,
                                    metric     = 'mean_squared_error',
                                    verbose    = False
                                )

    return metric, predictions
    

In [ ]:
# Function: simulate the daily steps of a participant based on a goal, their past behavior, and their sensitivity to rewards and punishments.

def simulate_daily_steps(goal, df_user, r, p, group_id, window=3):
    """
    Goal-centric, coherence-driven step simulator.
    Success depends primarily on (r,p)-sensitivity alignment.
    """

    # --------------------------------------------------
    # 1. Baseline (slow trend for realism)
    # --------------------------------------------------
    if len(df_user) < window:
        baseline = 6000
    else:
        baseline = df_user["steps"].tail(window).mean()

    # --------------------------------------------------
    # 2. Coherence score ∈ [-1, +1]
    # --------------------------------------------------
    if group_id == "g01":          # reward-sensitive
        coherence = (r - 0.6) / 0.4   # r > 0.6 → positive

    elif group_id == "g02":        # punishment-sensitive
        coherence = (p - 0.6) / 0.4   # p > 0.6 → positive

    coherence = max(-1.0, min(coherence, 1.0))

    # --------------------------------------------------
    # 3. Determine success margin
    # --------------------------------------------------
    # Coherent → above goal
    # Incoherent → below goal
    margin = coherence * 0.15 * goal   # max ±15%

    steps_today = goal + margin

    # --------------------------------------------------
    # 4. Minimal deterministic noise
    # --------------------------------------------------
    drift = (baseline - goal) * 0.05
    steps_today += drift

    # Clamp
    steps_today = max(1000, min(steps_today, goal * 1.2))

    # --------------------------------------------------
    # 5. Update baseline
    # --------------------------------------------------
    success = int(steps_today >= goal)

    if success:
        baseline += 0.03 * goal
    else:
        baseline -= 0.02 * goal

    baseline = max(3000, min(baseline, 15000))

    return int(steps_today), int(baseline)


In [7]:
# ------------
# PARAMETERS 
# ------------
n_participants = 30 
n_groups = 2
days = 181

np.random.seed(42)

# Create the dataframe simulating participants' daily steps for 6 months
df = dataframe_creation(n_participants, n_groups, days)
print(df.head())

df_full = df.copy()
# Keeps only the first 90 days for training and testing, simulating a real scenario where we have a limited number of dara and no future data  
df = df[df['date'] <= '2025-03-01']

        date participant_id group_id  baseline  steps
0 2025-01-01            p01      g01      7696   8692
1 2025-01-02            p01      g01      8194   7810
2 2025-01-03            p01      g01      7667  10088
3 2025-01-04            p01      g01      7968   7219
4 2025-01-05            p01      g01      7900   7167


In [ ]:
## ======================================================
# Q-learning functions
## ======================================================


# Function: initialize Q-table 

## Q-table structure:
## - rows: states (0: goal not reached, 1: goal reached)
## - columns: actions (reward_prob, punishment_prob)

def init_q_table():
    q = pd.DataFrame(
        0.0, 
        index=[0,1],       
        columns=[str(a) for a in ACTIONS]
    )
    return q


# ======================================================

# Function: choose action with epsilon-greedy strategy, guided by the last success history and current reward probability.

def choose_action_guided_pro(q_table, state, r_current, success_history, epsilon, explore_scale=0.12):
    
    if len(success_history) == 0:
        last_success = 0   
    else:
        last_success = success_history[-1]

    
    if np.random.rand() < epsilon:

        if last_success == 1:
            # if user had success → reward → push r up
            mean_r = min(1.0, r_current + 0.10)
        else:
            # if user failed → punishment → push p up → r down
            mean_r = max(0.0, r_current - 0.10)

        # sample around mean_r
        sampled_r = np.random.normal(loc=mean_r, scale=explore_scale)
        sampled_r = float(max(0.0, min(1.0, sampled_r)))

        # find closest action in ACTIONS
        r_values = [a[0] for a in ACTIONS]
        idx = int(np.argmin([abs(rv - sampled_r) for rv in r_values]))
        return ACTIONS[idx]

    else:
        # ============================
        # EXPLOITATION = argmax(Q)
        # ============================
        q_row = q_table.loc[state]
        a_str = q_row.idxmax()
        return eval(a_str)


# ======================================================

# Bellman Q-update function: Q(s,a) <- Q(s,a) + α [ r + γ max_{a'} Q(s',a') - Q(s,a) ]

def q_update(q_table, state, action, reward, next_state, alpha=0.1, gamma=0.9):
    action_key = str(action)  

    old_q = q_table.loc[state, action_key]
    next_max = q_table.loc[next_state].max()

    new_q = old_q + alpha * (reward + gamma * next_max - old_q)

    q_table.loc[state, action_key] = new_q

    return q_table


# ======================================================

# Function: compute reward based on steps taken, goal, and the current reward and punishment probabilities.

def compute_reward(steps, goal, r, p):

    margin = (steps - goal) / goal  

    if steps >= goal:
        return 1 + r * margin
    else:
        return - 1 + p * margin


In [ ]:
####### SIMULATION 1 - REINFORCEMENT LEARNING WITH Q-LEARNING AND GUIDED EXPLORATION ########

# ======================================================
#   RL PARAMETERS
# ======================================================
epsilon = 0.2
alpha = 0.1
gamma = 0.9
lags = 3
steps_ahead = 1
n_days_initial = 59
N_SIM_DAYS = 100

os.makedirs("users_history_RL", exist_ok=True)
os.makedirs("qtables_RL", exist_ok=True)

all_qtables = []

for user_id in df["participant_id"].unique():

    print(f"\n=== Simulating user {user_id} ===")

    df_user = df[df["participant_id"] == user_id].copy()
    df_user["date"] = pd.to_datetime(df_user["date"])   

    regressor = LinearRegression()
    n_days = n_days_initial

    group_id = df_user["group_id"].iloc[0]

    q_table = init_q_table()
    success_history = []
    history = []

    r_current = 0.5
    p_current = 0.5

    for day in range(N_SIM_DAYS):

        # ML prediction
        metric, prediction = model_training_prediction(
            df_user,
            user_id,
            regressor,
            lags,
            steps_ahead,
            n_days
        )
        goal = int(prediction["pred"].iloc[0])

        last_day = df_user.iloc[-1]

        if history is not None and len(history) > 0 and history[-1] is not None:
            last_goal = history[-1]["goal"]
            last_success = int(last_day["steps"] >= last_goal)
        else:
            last_success = int(last_day["steps"] >= last_day["baseline"])

        state = last_success

        action = choose_action_guided_pro(q_table, state, r_current, success_history, epsilon, explore_scale=0.20)

        r, p = action

        new_steps, new_baseline = simulate_daily_steps(goal, df_user, r, p, group_id)

        success = int(new_steps >= goal)
        success_history.append(success)

        # reward RL
        reward = compute_reward(new_steps, goal, r, p)

        # RL update
        q_table = q_update(q_table, state, action, reward, success, alpha, gamma)
        
        q_value = q_table.loc[state, str(action)]

        new_date = df_user["date"].max() + pd.Timedelta(days=1)
        df_user.loc[len(df_user)] = [new_date, user_id, group_id, new_baseline, new_steps]

        n_days += 1

        history.append({
            "day": day,
            "date": new_date,
            "goal": goal,
            "steps": new_steps,
            "action_r": r,
            "action_p": p,
            "success": success,
            "reward": reward,
            "q_value": q_value
        })
        

    # ======================================================
    #   SAVE USER HISTORY
    # ======================================================
    history_df = pd.DataFrame(history)
    history_df.to_excel(f"users_history_RL/history_{user_id}_{group_id}.xlsx", index=False)
    print(f"Saved history for {user_id}")

    # ======================================================
    #   SAVE FINAL Q-TABLE FOR USER
    # ======================================================
    q_flat = q_table.copy()
    q_flat["user_id"] = user_id
    q_flat["group_id"] = group_id
    all_qtables.append(q_flat)


# ======================================================
#   SAVE COMBINED Q-TABLES FOR ALL USERS
# ======================================================
qtables_all_df = pd.concat(all_qtables)
qtables_all_df.to_excel("qtables_RL/all_qtables.xlsx", index=True)
print("\nSaved combined Q-tables for all users.")



=== Simulating user p01 ===


100%|██████████| 1/1 [00:00<00:00, 315.72it/s]


Saved history for p01

=== Simulating user p02 ===


100%|██████████| 1/1 [00:00<00:00, 425.77it/s]


Saved history for p02

=== Simulating user p03 ===


...

Saved history for p28

=== Simulating user p29 ===


100%|██████████| 1/1 [00:00<00:00, 334.39it/s]


Saved history for p29

=== Simulating user p30 ===


100%|██████████| 1/1 [00:00<00:00, 330.60it/s]


Saved history for p30

Saved combined Q-tables for all users.


In [8]:
# ==========================
# ThompsonBandit class
# ==========================

class ThompsonBandit:

    # Function: Thompson Sampling, (1, 1) priors
    def __init__(self, n_arms, prior_a=1.0, prior_b=1.0):
        self.n_arms = n_arms
        self.alpha = {i: prior_a for i in range(n_arms)}
        self.beta  = {i: prior_b for i in range(n_arms)}
    
    # =======================================================================================

    # Function: Select arm 
    def select_arm(self):
        samples = {i: np.random.beta(self.alpha[i], self.beta[i]) for i in range(self.n_arms)}
        best = max(samples, key=samples.get)
        return best

    # =======================================================================================

    # Function: Update parameters based on observed reward 
    def update(self, arm_idx, reward):
        if reward == 1:
            self.alpha[arm_idx] += 1
        else:
            self.beta[arm_idx] += 1



In [ ]:
####### SIMULATION 2 - MAB WITH THOMPSON SAMPLING ########

# ======================================================
#   PARAMETERS
# ======================================================
lags = 3
steps_ahead = 1
n_days_initial = 59
N_SIM_DAYS = 100

os.makedirs("users_history_BANDIT", exist_ok=True)
os.makedirs("alpha_beta_BANDIT", exist_ok=True)

all_bandits = []

for user_id in df["participant_id"].unique():

    print(f"\n=== Simulating user {user_id} ===")

    df_user = df[df["participant_id"] == user_id].copy()
    df_user["date"] = pd.to_datetime(df_user["date"])

    regressor = LinearRegression()
    n_days = n_days_initial

    group_id = df_user["group_id"].iloc[0]

    success_history = []
    history = []

    r_current = 0.5
    p_current = 0.5

    # -------------------------
    # Bandit for each state (success=0, success=1)
    # -------------------------
    n_arms = len(ACTIONS)
    bandit_per_state = {
        0: ThompsonBandit(n_arms=n_arms, prior_a=1.0, prior_b=1.0),
        1: ThompsonBandit(n_arms=n_arms, prior_a=1.0, prior_b=1.0),
    }

    for day in range(N_SIM_DAYS):

        # ML prediction
        metric, prediction = model_training_prediction(
            df_user,
            user_id,
            regressor,
            lags,
            steps_ahead,
            n_days
        )
        goal = int(prediction["pred"].iloc[0])

        last_day = df_user.iloc[-1]

        if history is not None and len(history) > 0 and history[-1] is not None:
            last_goal = history[-1]["goal"]
            last_success = int(last_day["steps"] >= last_goal)
        else:
            last_success = int(last_day["steps"] >= last_day["baseline"])

        state = last_success

        arm_idx = bandit_per_state[state].select_arm()
        r_action, p_action = ACTIONS[arm_idx]

        steps_today, baseline = simulate_daily_steps(goal, df_user, r_action, p_action, group_id)

        reward_ts = int(steps_today >= goal)
        bandit_per_state[state].update(arm_idx, reward_ts)

        next_date = df_user["date"].max() + pd.Timedelta(days=1)
        df_user.loc[len(df_user)] = [next_date, user_id, group_id, baseline, steps_today]

        n_days += 1

        success = int(steps_today >= goal)
        history.append({
            "day": day + 1,
            "date": next_date,
            "goal": goal,
            "steps": steps_today,
            "success": success,
            "state": state,
            "action_r": r_action,
            "action_p": p_action,
            "reward_ts": reward_ts
        })




    # ======================================================
    #   SAVE USER HISTORY
    # ======================================================
    history_df = pd.DataFrame(history)
    history_df.to_excel(f"users_history_BANDIT/history_{user_id}_{group_id}.xlsx", index=False)
    print(f"Saved history for {user_id}")
    
    # ======================================================
    #   SAVE ALPHA-BETA VALUES FOR ANALYSIS
    # ======================================================

    rows = []

    for state in [0, 1]:

        bandit = bandit_per_state[state]

        row = {
            "state": state,
            "user_id": user_id,
            "group_id": group_id,
        }

        for arm_idx, (r_val, p_val) in enumerate(ACTIONS):

            alpha = bandit.alpha[arm_idx]
            beta = bandit.beta[arm_idx]

            q_value = np.log(alpha / beta)

            row[f"({r_val},{p_val})"] = q_value

        rows.append(row)

    bandit_user_df = pd.DataFrame(rows)

    all_bandits.append(bandit_user_df)




=== Simulating user p01 ===


100%|██████████| 1/1 [00:00<00:00, 568.95it/s]


100%|██████████| 1/1 [00:00<00:00, 437.13it/s]


Saved history for p01

=== Simulating user p02 ===


100%|██████████| 1/1 [00:00<00:00, 422.05it/s]


...

Saved history for p28

=== Simulating user p29 ===


100%|██████████| 1/1 [00:00<00:00, 274.64it/s]


Saved history for p29

=== Simulating user p30 ===


100%|██████████| 1/1 [00:00<00:00, 257.22it/s]


Saved history for p30


In [9]:
# ======================================================
# Random
# ======================================================

# function that returns r and p randomly from the ACTIONS list
def random_action():
    return ACTIONS[np.random.randint(0, len(ACTIONS))]

In [ ]:
####### SIMULATION 3 - RANDOM ACTION SELECTION ########
 
# =====================================================
#  PARAMETERS
# =====================================================

lags = 3
steps_ahead = 1
n_days_initial = 59
N_SIM_DAYS = 100

os.makedirs("users_history_random", exist_ok=True)


for user_id in df["participant_id"].unique():

    print(f"\n=== Simulating user {user_id} ===")

    df_user = df[df["participant_id"] == user_id].copy()
    df_user["date"] = pd.to_datetime(df_user["date"])

    regressor = LinearRegression()
    n_days = n_days_initial

    group_id = df_user["group_id"].iloc[0]

    success_history = []
    history = []

    r_current = 0.5
    p_current = 0.5

    for day in range(N_SIM_DAYS):

        baseline = df_user["steps"].tail(7).mean()

        # ML prediction
        metric, prediction = model_training_prediction(
            df_user,
            user_id,
            regressor,
            lags,
            steps_ahead,
            n_days
        )
        goal = int(int(prediction["pred"].iloc[0]))

        action = random_action()

        r, p = action

        new_steps, baseline = simulate_daily_steps(goal, df_user, r, p, group_id)

        success = int(new_steps >= goal)
        success_history.append(success)

        new_date = df_user["date"].max() + pd.Timedelta(days=1)
        df_user.loc[len(df_user)] = [new_date, user_id, group_id, baseline, new_steps]

        n_days += 1

        history.append({
            "day": day,
            "date": new_date,
            "goal": goal,
            "steps": new_steps,
            "action_r": r,
            "action_p": p,
            "success": success
        })

        

    # ======================================================
    #   SAVE USER HISTORY
    # ======================================================
    history_df = pd.DataFrame(history)
    history_df.to_excel(f"users_history_random/history_{user_id}_{group_id}.xlsx", index=False)
    print(f"Saved history for {user_id}")




=== Simulating user p01 ===


100%|██████████| 1/1 [00:00<00:00, 415.57it/s]


Saved history for p01

=== Simulating user p02 ===


100%|██████████| 1/1 [00:00<00:00, 567.72it/s]


Saved history for p02

=== Simulating user p03 ===


...

Saved history for p28

=== Simulating user p29 ===


100%|██████████| 1/1 [00:00<00:00, 577.49it/s]


Saved history for p29

=== Simulating user p30 ===


100%|██████████| 1/1 [00:00<00:00, 824.03it/s]


Saved history for p30


In [17]:
####### ANALISI Q-TABLES SALVATE ########

# per ognu gruppo, media Q-table
for gid in df["group_id"].unique():
    group_qtables = qtables_all_df[qtables_all_df["group_id"] == gid]
    group_mean_qtable = group_qtables.drop(columns=["user_id", "group_id"]).mean()
    print(f"\nMean Q-table for group {gid}:")
    print(group_mean_qtable)


# Load all Q-tables
qtables_all = pd.read_excel(
    "qtables_RL/all_qtables.xlsx",
    index_col=0  # index = state
)

# Assicuriamoci che lo stato sia esplicito
qtables_all = qtables_all.reset_index().rename(columns={"index": "state"})

# Action columns are those that look like "(r, p)"
action_cols = [
    c for c in qtables_all.columns
    if c.startswith("(") and c.endswith(")")
]

group_state_action_mean = (
    qtables_all
    .groupby(["group_id", "state"])[action_cols]
    .mean()
    .reset_index()
)


for gid in sorted(group_state_action_mean["group_id"].unique()):
    print(f"\n========================")
    print(f"Group {gid}")
    print(f"========================")

    df_g = group_state_action_mean[group_state_action_mean["group_id"] == gid]

    for state in sorted(df_g["state"].unique()):
        print(f"\nState {state}:")
        row = df_g[df_g["state"] == state].iloc[0]

        for a in action_cols:
            print(f"  {a}: {row[a]:.6f}")




Mean Q-table for group g01:
(0.1, 0.9)   -0.116147
(0.2, 0.8)   -0.107537
(0.3, 0.7)   -0.114063
(0.4, 0.6)   -0.092670
(0.5, 0.5)   -0.089834
(0.6, 0.4)    0.629108
(0.7, 0.3)    1.918410
(0.8, 0.2)    0.575439
(0.9, 0.1)    0.558004
dtype: float64

Mean Q-table for group g02:
(0.1, 0.9)    3.575128
(0.2, 0.8)    0.067453
(0.3, 0.7)    0.285021
(0.4, 0.6)    0.586609
(0.5, 0.5)    0.014796
(0.6, 0.4)    0.016345
(0.7, 0.3)   -0.002476
(0.8, 0.2)    0.005264
(0.9, 0.1)   -0.000846
dtype: float64

Group g01

State 0:
  (0.1, 0.9): -0.123930
  (0.2, 0.8): -0.138912
  (0.3, 0.7): -0.150790
  (0.4, 0.6): -0.108585
  (0.5, 0.5): -0.105016
  (0.6, 0.4): -0.110646
  (0.7, 0.3): 1.786215
  (0.8, 0.2): 0.007067
  (0.9, 0.1): 0.000000

State 1:
  (0.1, 0.9): -0.108363
  (0.2, 0.8): -0.076162
  (0.3, 0.7): -0.077335
  (0.4, 0.6): -0.076755
  (0.5, 0.5): -0.074652
  (0.6, 0.4): 1.368862
  (0.7, 0.3): 2.050605
  (0.8, 0.2): 1.143810
  (0.9, 0.1): 1.116009

Group g02

State 0:
  (0.1, 0.9): 2.02196

In [20]:
# ======================================================
#   SALVATAGGIO EXCEL UNICO CON TUTTI GLI UTENTI
# ======================================================

final_bandit_df = pd.concat(all_bandits, ignore_index=True)

# riordino colonne come Q-table
final_bandit_df = final_bandit_df[
    ["state"]
    + [f"({r},{p})" for r, p in ACTIONS]
    + ["user_id", "group_id"]
]

final_bandit_df.to_excel(
    "alpha_beta_BANDIT/bandit_tables_all_users.xlsx",
    index=False
)

print("Saved bandit_tables_all_users.xlsx")


def beta_to_q(alpha, beta, eps=1e-6):
    return np.log((alpha + eps) / (beta + eps))

rows = []

for user_id in final_bandit_df["user_id"].unique():

    df_u = final_bandit_df[final_bandit_df["user_id"] == user_id]

    row = {
        "user_id": user_id,
        "group_id": df_u["group_id"].iloc[0]
    }

    for r, p in ACTIONS:
        col = f"({r},{p})"

        # media sui due stati
        q_vals = df_u[col].values
        row[col] = np.mean(q_vals)

    rows.append(row)

user_q_like_df = pd.DataFrame(rows)

action_cols = [f"({r},{p})" for r, p in ACTIONS]

for gid in user_q_like_df["group_id"].unique():

    df_g = user_q_like_df[user_q_like_df["group_id"] == gid]

    mean_q = df_g[action_cols].mean()

    print(f"\nMean Q-table for group {gid}:")
    print(mean_q)


action_cols = [f"({r},{p})" for r, p in ACTIONS]

for gid in final_bandit_df["group_id"].unique():

    print(f"\n===== GROUP {gid} =====")

    df_group = final_bandit_df[final_bandit_df["group_id"] == gid]

    for state in [0, 1]:

        df_state = df_group[df_group["state"] == state]

        mean_q = df_state[action_cols].mean()

        print(f"\nMean Q-table – state {state}")
        print(mean_q)




Saved bandit_tables_all_users.xlsx

Mean Q-table for group g01:
(0.1,0.9)   -0.618169
(0.2,0.8)   -0.598990
(0.3,0.7)   -0.591138
(0.4,0.6)   -0.654790
(0.5,0.5)   -0.718441
(0.6,0.4)    0.454770
(0.7,0.3)    1.979634
(0.8,0.2)    1.613368
(0.9,0.1)    2.122280
dtype: float64

Mean Q-table for group g02:
(0.1,0.9)    1.940424
(0.2,0.8)    1.984374
(0.3,0.7)    1.895274
(0.4,0.6)    0.261423
(0.5,0.5)   -0.521823
(0.6,0.4)   -0.687484
(0.7,0.3)   -0.677894
(0.8,0.2)   -0.668305
(0.9,0.1)   -0.598990
dtype: float64

===== GROUP g01 =====

Mean Q-table – state 0
(0.1,0.9)   -0.462098
(0.2,0.8)   -0.450772
(0.3,0.7)   -0.535339
(0.4,0.6)   -0.562370
(0.5,0.5)   -0.681821
(0.6,0.4)   -0.516160
(0.7,0.3)    1.111535
(0.8,0.2)    0.755061
(0.9,0.1)    1.076949
dtype: float64

Mean Q-table – state 1
(0.1,0.9)   -0.774240
(0.2,0.8)   -0.747209
(0.3,0.7)   -0.646937
(0.4,0.6)   -0.747209
(0.5,0.5)   -0.755061
(0.6,0.4)    1.425701
(0.7,0.3)    2.847732
(0.8,0.2)    2.471675
(0.9,0.1)    3.167611